# Structured Streaming in Databricks

**Structured Streaming** is a scalable and fault-tolerant stream processing engine built on Spark SQL. It allows you to process real-time data streams using the same DataFrame/Dataset API as batch processing.

## Key Concepts

* **Streaming DataFrame**: Represents unbounded data streams as a table that continuously grows
* **Input Sources**: rate, files, Kafka, Delta tables, Auto Loader
* **Transformations**: Same DataFrame operations as batch (select, filter, groupBy, etc.)
* **Output Sinks**: console, memory, files, Delta tables, Kafka
* **Triggers**: Define when to process data (processing time, once, continuous)
* **Checkpointing**: Fault tolerance through write-ahead logs

---

## Streaming Processing Model

```
Input Stream → Transformations → Output Sink
     ↓              ↓                ↓
  Source         Operations        Destination
 (Kafka/Files)  (Filter/Agg)    (Delta/Files)
```

---

## Key Features

✅ **Exactly-once processing** — No duplicates or data loss  
✅ **Fault tolerance** — Automatic recovery from failures  
✅ **Unified API** — Same code for batch and streaming  
✅ **Scalability** — Handle millions of events per second  
✅ **Late data handling** — Watermarking for out-of-order events  
✅ **Stateful operations** — Aggregations, joins, deduplication  

---

## Output Modes

| Mode | Description | Use Case |
|------|-------------|----------|
| **Append** | Only new rows added to result | ETL, data ingestion |
| **Complete** | Entire result rewritten | Small aggregations, dashboards |
| **Update** | Only changed rows updated | Aggregations with updates |

---

Let's explore with hands-on examples!

In [0]:
# Structured Streaming Basics: Rate Source
# Rate source generates synthetic data for testing

from pyspark.sql.functions import *

# Create a streaming DataFrame from rate source
# Generates rows with timestamp and value columns
rate_stream = (spark
    .readStream
    .format("rate")
    .option("rowsPerSecond", 10)  # Generate 10 rows per second
    .option("numPartitions", 2)    # Number of partitions
    .load()
)

# Display the schema
print("Rate Stream Schema:")
rate_stream.printSchema()

# Show what columns are available
print("\nColumns:", rate_stream.columns)
print("\n✅ Rate source is great for testing streaming logic without external dependencies!")

In [0]:
# Write streaming data to console for debugging
# This is useful for development and testing

from pyspark.sql.functions import *

# Create streaming DataFrame
rate_stream = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

# Add transformations
transformed = rate_stream.select(
    col("timestamp"),
    col("value"),
    (col("value") * 2).alias("doubled"),
    (col("value") % 2 == 0).alias("is_even")
)

# Write to console (for demo only - stops automatically after a few batches)
query = (transformed
    .writeStream
    .format("console")
    .outputMode("append")  # Only show new rows
    .option("truncate", False)
    .option("numRows", 10)
    .start()
)

print(f"\n✅ Streaming query started with ID: {query.id}")
print(f"Status: {query.status}")
print("\nNote: Console sink is for debugging only. Check output below.")

# Let it run for a short time
import time
time.sleep(10)

# Stop the query
query.stop()
print("\n✅ Query stopped successfully!")

In [0]:
# Streaming aggregations with time windows
# Windows allow you to group events by time periods

from pyspark.sql.functions import *

# Create rate stream
rate_stream = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 20)
    .load()
)

# Add a category column for demonstration
stream_with_category = rate_stream.withColumn(
    "category",
    when(col("value") % 3 == 0, "A")
    .when(col("value") % 3 == 1, "B")
    .otherwise("C")
)

# Perform windowed aggregation
# Tumbling window: non-overlapping 10-second windows
windowed_counts = (stream_with_category
    .groupBy(
        window(col("timestamp"), "10 seconds"),  # 10-second windows
        col("category")
    )
    .agg(
        count("*").alias("count"),
        avg("value").alias("avg_value"),
        min("value").alias("min_value"),
        max("value").alias("max_value")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("category"),
        col("count"),
        round(col("avg_value"), 2).alias("avg_value"),
        col("min_value"),
        col("max_value")
    )
    .orderBy("window_start", "category")
)

print("\n✅ Windowed aggregation query created!")
print("This groups events into 10-second tumbling windows and calculates stats per category.")
print("\nSchema:")
windowed_counts.printSchema()

In [0]:
# Write streaming data to Delta table
# Delta provides ACID transactions and time travel for streaming data

from pyspark.sql.functions import *

# Define checkpoint and output locations
checkpoint_location = "/tmp/streaming_demo/checkpoint_append"
output_path = "/tmp/streaming_demo/delta_output_append"

# Create streaming source
rate_stream = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 10)
    .load()
)

# Add some transformations
enriched_stream = rate_stream.select(
    col("timestamp"),
    col("value"),
    (col("value") * 2).alias("doubled"),
    current_timestamp().alias("processing_time"),
    date_format(col("timestamp"), "yyyy-MM-dd").alias("date"),
    hour(col("timestamp")).alias("hour")
)

# Write to Delta table in append mode
query = (enriched_stream
    .writeStream
    .format("delta")
    .outputMode("append")  # Only new rows
    .option("checkpointLocation", checkpoint_location)
    .option("path", output_path)
    .trigger(processingTime="5 seconds")  # Process every 5 seconds
    .start()
)

print(f"\n✅ Streaming to Delta table started!")
print(f"Query ID: {query.id}")
print(f"Output path: {output_path}")
print(f"Checkpoint: {checkpoint_location}")
print("\nLet it run for 15 seconds...")

# Let it run
import time
time.sleep(15)

# Stop the stream
query.stop()
print("\n✅ Stream stopped!")

# Read the Delta table to verify data was written
df = spark.read.format("delta").load(output_path)
print(f"\nTotal rows written: {df.count()}")
print("\nSample data:")
df.orderBy("timestamp").show(10, truncate=False)

In [0]:
# Streaming from file sources (JSON, CSV, Parquet)
# Monitors a directory and processes new files as they arrive

from pyspark.sql.functions import *
from pyspark.sql.types import *

# Define schema for incoming JSON files
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("timestamp", TimestampType(), True)
])

# Create input directory for demo
input_path = "/tmp/streaming_demo/input_json"
dbutils.fs.mkdirs(input_path)

print(f"Input directory: {input_path}")
print("\nCreating sample JSON files...")

# Write sample JSON files to simulate incoming data
import json
import datetime

for batch in range(3):
    data = [{
        "id": i + (batch * 10),
        "name": f"user_{i + (batch * 10)}",
        "value": (i + (batch * 10)) * 1.5,
        "timestamp": (datetime.datetime.now() + datetime.timedelta(seconds=i)).isoformat()
    } for i in range(10)]
    
    json_str = "\n".join([json.dumps(record) for record in data])
    dbutils.fs.put(f"{input_path}/batch_{batch}.json", json_str, overwrite=True)

print(f"\n✅ Created 3 JSON files with sample data")

# Create streaming DataFrame from JSON files
file_stream = (spark.readStream
    .format("json")
    .schema(schema)
    .option("maxFilesPerTrigger", 1)  # Process 1 file per trigger
    .load(input_path)
)

print("\n✅ File stream configured!")
print("Schema:")
file_stream.printSchema()

In [0]:
# Watermarking: Handle late-arriving data in streaming aggregations
# Watermark = threshold for how late data can arrive

from pyspark.sql.functions import *

# Create rate stream with event time
rate_stream = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 10)
    .load()
)

# Add some artificial late data by adjusting timestamps
stream_with_delays = rate_stream.select(
    # Randomly delay some events by 0-30 seconds
    (col("timestamp") - expr("INTERVAL '0' SECOND")).alias("event_time"),
    col("value"),
    col("timestamp").alias("arrival_time")
)

# Aggregation with watermark
# Watermark of 10 seconds means:
# - Events up to 10 seconds late will be included
# - Events more than 10 seconds late will be dropped
watermarked_agg = (stream_with_delays
    .withWatermark("event_time", "10 seconds")  # Watermark specification
    .groupBy(
        window(col("event_time"), "5 seconds"),  # 5-second windows
    )
    .agg(
        count("*").alias("event_count"),
        avg("value").alias("avg_value")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("event_count"),
        round(col("avg_value"), 2).alias("avg_value")
    )
)

print("\n✅ Watermarked aggregation query created!")
print("\nWatermark: 10 seconds")
print("Window size: 5 seconds")
print("\nHow it works:")
print("1. Events are grouped into 5-second windows based on event_time")
print("2. Windows are kept open for 10 seconds after the latest event")
print("3. Events arriving >10 seconds late are dropped")
print("4. This prevents unbounded state growth")

In [0]:
# Different output modes for streaming queries
# - Append: Only new rows (default, best for ETL)
# - Complete: Entire result table (for small aggregations)
# - Update: Only changed rows (for aggregations with updates)

from pyspark.sql.functions import *

# Create streaming source
rate_stream = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 10)
    .load()
)

# Create aggregation
agg_query = (rate_stream
    .withColumn("category", (col("value") % 3).cast("string"))
    .groupBy("category")
    .agg(
        count("*").alias("count"),
        avg("value").alias("avg_value")
    )
)

print("\n=== Output Modes Comparison ===")
print("\n1. APPEND Mode:")
print("   - Only works with non-aggregated queries or aggregations with watermark")
print("   - Outputs only new rows")
print("   - Most efficient for ETL pipelines")
print("   - Example: Raw data ingestion, event logging")

print("\n2. COMPLETE Mode:")
print("   - Outputs the entire result table every trigger")
print("   - Works with all aggregations")
print("   - Use for small result sets (dashboards, monitoring)")
print("   - Example: Real-time KPI dashboards")

print("\n3. UPDATE Mode:")
print("   - Outputs only rows that changed since last trigger")
print("   - More efficient than complete for large result sets")
print("   - Use with aggregations when you need incremental updates")
print("   - Example: Updating aggregated metrics")

print("\n✅ Choose the right mode based on your use case!")

# Example: Write with COMPLETE mode to memory sink for inspection
query = (agg_query
    .writeStream
    .format("memory")
    .queryName("streaming_agg_complete")  # Table name for SQL queries
    .outputMode("complete")  # Complete mode
    .start()
)

import time
time.sleep(5)

print("\nQuerying the in-memory table:")
spark.sql("SELECT * FROM streaming_agg_complete ORDER BY category").show()

query.stop()
print("\n✅ Query stopped!")

In [0]:
# Trigger modes control when streaming queries process data
# Different modes offer different latency vs throughput trade-offs

from pyspark.sql.functions import *

print("=== Trigger Modes in Structured Streaming ===")
print("\n1. PROCESSING TIME (Micro-batch)")
print("   .trigger(processingTime='5 seconds')")
print("   - Process data every N seconds/minutes")
print("   - Most common mode, good for most use cases")
print("   - Balances latency and throughput")
print("   - Example: .trigger(processingTime='10 seconds')")

print("\n2. ONCE (Single Batch)")
print("   .trigger(once=True)")
print("   - Process all available data and stop")
print("   - Great for scheduled jobs, testing")
print("   - Use with job schedulers for cost efficiency")
print("   - Example: Daily batch processing of streaming data")

print("\n3. AVAILABLE NOW (Multi-batch processing)")
print("   .trigger(availableNow=True)")
print("   - Process all available data in multiple batches and stop")
print("   - Better than 'once' for large data volumes")
print("   - Processes in chunks, more memory efficient")
print("   - Example: Backfilling historical data")

print("\n4. CONTINUOUS (Experimental)")
print("   .trigger(continuous='1 second')")
print("   - Ultra-low latency (~1ms)")
print("   - Limited operations supported")
print("   - Use for time-critical applications")
print("   - Example: Real-time alerting systems")

print("\n5. DEFAULT (No trigger specified)")
print("   .start()  # No trigger specified")
print("   - Processes as soon as previous batch completes")
print("   - Maximum throughput, minimum latency")
print("   - Can consume more resources")

# Example: Processing time trigger
rate_stream = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 100)
    .load()
)

query = (rate_stream
    .writeStream
    .format("console")
    .trigger(processingTime="3 seconds")  # Process every 3 seconds
    .start()
)

print("\n✅ Started streaming with 3-second processing time trigger")
import time
time.sleep(10)
query.stop()
print("✅ Stopped!")

In [0]:
# Deduplication in streaming queries
# Remove duplicate records based on key columns

from pyspark.sql.functions import *

# Create streaming source with potential duplicates
rate_stream = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 10)
    .load()
)

# Create data with IDs (some will be duplicates)
stream_with_ids = rate_stream.select(
    (col("value") % 50).alias("id"),  # IDs from 0-49 (duplicates will occur)
    col("timestamp"),
    col("value")
)

print("\n=== Streaming Deduplication ===")
print("\nWithout watermark: Keeps ALL historical IDs in state (memory grows unbounded)")
print("With watermark: Drops state for old IDs after watermark threshold")

# Deduplication WITHOUT watermark (unbounded state growth)
dedup_unbounded = stream_with_ids.dropDuplicates(["id"])

print("\n⚠️ Unbounded deduplication: Keeps all IDs forever (not recommended for production)")

# Deduplication WITH watermark (bounded state)
dedup_bounded = (stream_with_ids
    .withWatermark("timestamp", "1 minute")  # Keep state for 1 minute
    .dropDuplicates(["id", "timestamp"])     # Deduplicate on ID within time window
)

print("✅ Bounded deduplication: Drops state after 1 minute (production-ready)")

# Run bounded deduplication query
output_path = "/tmp/streaming_demo/dedup_output"
checkpoint_path = "/tmp/streaming_demo/dedup_checkpoint"

query = (dedup_bounded
    .select("id", "timestamp", "value")
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("path", output_path)
    .trigger(processingTime="5 seconds")
    .start()
)

print(f"\n✅ Deduplication query started!")
print(f"Output: {output_path}")

import time
time.sleep(15)

query.stop()
print("\n✅ Query stopped!")

# Check results
df = spark.read.format("delta").load(output_path)
print(f"\nTotal unique records: {df.count()}")
print(f"Unique IDs: {df.select('id').distinct().count()}")
print("\nSample data:")
df.orderBy("id", "timestamp").show(10)

In [0]:
# Monitor and manage streaming queries
# Get status, metrics, and control query execution

from pyspark.sql.functions import *

# Start a streaming query
rate_stream = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 50)
    .load()
)

query = (rate_stream
    .select(
        col("timestamp"),
        col("value"),
        (col("value") * 2).alias("doubled")
    )
    .writeStream
    .format("memory")
    .queryName("monitoring_demo")
    .outputMode("append")
    .trigger(processingTime="2 seconds")
    .start()
)

print("\n=== Streaming Query Monitoring ===")

# 1. Query metadata
print(f"\n1. QUERY METADATA")
print(f"   Query ID: {query.id}")
print(f"   Run ID: {query.runId}")
print(f"   Name: {query.name}")

# 2. Query status
import time
time.sleep(3)

status = query.status
print(f"\n2. QUERY STATUS")
print(f"   Is active: {query.isActive}")
print(f"   Message: {status['message']}")
print(f"   Is data available: {status['isDataAvailable']}")
print(f"   Is trigger active: {status['isTriggerActive']}")

# 3. Progress information (recent batch metrics)
time.sleep(5)

if query.lastProgress:
    progress = query.lastProgress
    print(f"\n3. LAST BATCH METRICS")
    print(f"   Batch ID: {progress['batchId']}")
    print(f"   Timestamp: {progress['timestamp']}")
    print(f"   Input rows: {progress['numInputRows']}")
    
    if 'sources' in progress and len(progress['sources']) > 0:
        source = progress['sources'][0]
        print(f"   Start offset: {source.get('startOffset', 'N/A')}")
        print(f"   End offset: {source.get('endOffset', 'N/A')}")
    
    if 'sink' in progress:
        print(f"   Sink description: {progress['sink']['description']}")

# 4. List all active streaming queries
print(f"\n4. ALL ACTIVE QUERIES")
for active_query in spark.streams.active:
    print(f"   - {active_query.name} (ID: {active_query.id})")

# 5. Wait for termination (with timeout)
print(f"\n5. WAITING FOR QUERY (5 seconds)...")
try:
    query.awaitTermination(timeout=5)  # Wait max 5 seconds
except:
    pass

# 6. Stop the query
query.stop()
print(f"\n✅ Query stopped!")
print(f"   Final status: {query.status}")

# 7. Exception handling
if query.exception():
    print(f"\n⚠️ Query had an exception: {query.exception()}")
else:
    print(f"\n✅ No exceptions during execution")

In [0]:
# Stream-Stream Joins: Join two streaming DataFrames
# Requires watermarks on both sides to bound state

from pyspark.sql.functions import *
from pyspark.sql.types import *

print("\n=== Stream-Stream Joins ===")

# Stream 1: Orders
orders_stream = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
    .select(
        col("value").alias("order_id"),
        col("timestamp").alias("order_time"),
        (col("value") % 100).alias("customer_id"),
        (col("value") * 10.5).alias("amount")
    )
    .withWatermark("order_time", "10 seconds")
)

# Stream 2: Customers (with slight delay)
from pyspark.sql.functions import expr

customers_stream = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .option("numPartitions", 1)
    .load()
    .select(
        (col("value") % 100).alias("customer_id"),
        col("timestamp").alias("customer_time"),
        concat(lit("Customer_"), col("value") % 100).alias("customer_name"),
        (col("value") % 5).alias("tier")  # 0-4 tier levels
    )
    .withWatermark("customer_time", "10 seconds")
)

print("\nCreated two streams:")
print("1. Orders: order_id, order_time, customer_id, amount")
print("2. Customers: customer_id, customer_time, customer_name, tier")
print("\nBoth have 10-second watermarks")

# Inner join with time constraint
# Join condition includes time window to limit state
joined_stream = orders_stream.join(
    customers_stream,
    expr("""
        orders_stream.customer_id = customers_stream.customer_id AND
        orders_stream.order_time >= customers_stream.customer_time AND
        orders_stream.order_time <= customers_stream.customer_time + interval 1 minute
    """),
    "inner"
).select(
    col("order_id"),
    col("order_time"),
    col("customer_name"),
    col("tier"),
    round(col("amount"), 2).alias("amount")
)

print("\n✅ Stream-stream join configured!")
print("Join type: INNER")
print("Time constraint: Orders must be within 1 minute of customer event")
print("\nNote: Stream-stream joins require:")
print("  1. Watermarks on both streams")
print("  2. Time-bound join condition")
print("  3. Join keys + time constraint")

# Write joined stream
query = (joined_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .trigger(processingTime="5 seconds")
    .start()
)

import time
time.sleep(15)
query.stop()
print("\n✅ Join query stopped!")

In [0]:
# Streaming from Delta tables
# Process incremental changes efficiently

from pyspark.sql.functions import *

print("\n=== Streaming from Delta Tables ===")

# First, create a Delta table to stream from
source_table_path = "/tmp/streaming_demo/source_delta_table"

# Create initial data
initial_data = spark.range(0, 100).select(
    col("id"),
    (col("id") * 2).alias("value"),
    current_timestamp().alias("created_at")
)

initial_data.write.format("delta").mode("overwrite").save(source_table_path)
print(f"✅ Created Delta table at: {source_table_path}")
print(f"Initial row count: {initial_data.count()}")

# Stream from the Delta table
delta_stream = (spark.readStream
    .format("delta")
    .load(source_table_path)
)

print("\n✅ Delta stream created!")
print("Schema:")
delta_stream.printSchema()

# Process the stream
processed = delta_stream.select(
    col("id"),
    col("value"),
    col("created_at"),
    (col("value") * 1.5).alias("processed_value"),
    current_timestamp().alias("processed_at")
)

# Write to another Delta table
output_path = "/tmp/streaming_demo/processed_delta"
checkpoint_path = "/tmp/streaming_demo/delta_checkpoint"

query = (processed
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("path", output_path)
    .trigger(processingTime="5 seconds")
    .start()
)

print(f"\n✅ Streaming query started!")
print(f"Reading from: {source_table_path}")
print(f"Writing to: {output_path}")

import time
time.sleep(5)

# Append more data to source table (simulating incremental updates)
print("\nAppending new data to source table...")
new_data = spark.range(100, 150).select(
    col("id"),
    (col("id") * 2).alias("value"),
    current_timestamp().alias("created_at")
)
new_data.write.format("delta").mode("append").save(source_table_path)
print("✅ Appended 50 more rows")

# Let stream process the new data
time.sleep(10)

query.stop()
print("\n✅ Stream stopped!")

# Verify results
result_df = spark.read.format("delta").load(output_path)
print(f"\nTotal processed rows: {result_df.count()}")
print("\nSample processed data:")
result_df.orderBy("id").show(10)

print("\n✅ Delta-to-Delta streaming complete!")
print("\nBenefits of Delta streaming:")
print("  • ACID transactions")
print("  • Schema evolution")
print("  • Time travel")
print("  • Efficient incremental processing")
print("  • Change Data Feed (CDF) support")

## Structured Streaming - Complete Reference

---

### Core Concepts

#### Streaming DataFrame
* Unbounded table that continuously grows
* Same API as batch DataFrames
* Incremental query execution
* Fault-tolerant and exactly-once semantics

#### Programming Model
```python
# Read Stream
df = spark.readStream
    .format("source")
    .option("key", "value")
    .load()

# Transform
transformed = df.select(...).filter(...).groupBy(...)

# Write Stream
query = transformed.writeStream
    .format("sink")
    .outputMode("mode")
    .trigger(processingTime="interval")
    .option("checkpointLocation", "path")
    .start()
```

---

### Input Sources

| Source | Format | Use Case | Options |
|--------|--------|----------|--------|
| **Rate** | `rate` | Testing, demos | `rowsPerSecond`, `numPartitions` |
| **Files** | `json`, `csv`, `parquet`, `orc` | Batch file ingestion | `maxFilesPerTrigger`, `latestFirst` |
| **Kafka** | `kafka` | Event streaming | `kafka.bootstrap.servers`, `subscribe` |
| **Delta** | `delta` | Delta Lake tables | `ignoreChanges`, `ignoreDeletes` |
| **Auto Loader** | `cloudFiles` | Scalable file ingestion | `cloudFiles.format`, `cloudFiles.schemaLocation` |
| **Socket** | `socket` | Development only | `host`, `port` |

---

### Output Sinks

| Sink | Format | Use Case | Modes Supported |
|------|--------|----------|----------------|
| **Console** | `console` | Debugging | Append, Complete, Update |
| **Memory** | `memory` | Testing, inspection | Append, Complete |
| **Files** | `json`, `csv`, `parquet`, `orc` | Data lake storage | Append |
| **Delta** | `delta` | ACID table storage | Append, Complete, Update |
| **Kafka** | `kafka` | Event publishing | Append, Update |
| **Foreach** | Custom | Custom logic | Append, Update |
| **ForeachBatch** | Custom | Batch-level processing | All modes |

---

### Output Modes

#### Append Mode (Default)
```python
.outputMode("append")
```
* Only new rows added since last trigger
* Cannot use with aggregations without watermark
* Most efficient for ETL pipelines
* **Use for**: Data ingestion, event logging

#### Complete Mode
```python
.outputMode("complete")
```
* Entire result table rewritten every trigger
* Works with all aggregations
* High memory/storage requirements
* **Use for**: Small aggregations, dashboards

#### Update Mode
```python
.outputMode("update")
```
* Only rows that changed since last trigger
* More efficient than complete for large results
* Works with aggregations
* **Use for**: Incremental metric updates

---

### Trigger Modes

#### Processing Time (Micro-batch)
```python
.trigger(processingTime="10 seconds")
```
* Process every N seconds/minutes
* Balances latency and throughput
* Most common mode

#### Once (Single Trigger)
```python
.trigger(once=True)
```
* Process all available data once and stop
* Great for scheduled jobs
* Cost-efficient for batch-like streaming

#### Available Now (Multi-batch)
```python
.trigger(availableNow=True)
```
* Process all data in multiple batches
* Better than `once` for large volumes
* Memory efficient backfilling

#### Continuous (Experimental)
```python
.trigger(continuous="1 second")
```
* Ultra-low latency (~1ms)
* Limited operations
* Experimental feature

#### Default (Immediate)
```python
.start()  # No trigger specified
```
* Process as fast as possible
* Maximum throughput
* Can consume more resources

---

### Windowing Operations

#### Tumbling Windows
```python
# Non-overlapping fixed-size windows
.groupBy(window(col("timestamp"), "10 minutes"))
```

#### Sliding Windows
```python
# Overlapping windows
.groupBy(window(col("timestamp"), "10 minutes", "5 minutes"))
# Window size: 10 min, Slide interval: 5 min
```

#### Session Windows
```python
# Windows based on session gaps
.groupBy(session_window(col("timestamp"), "30 minutes"))
```

---

### Watermarking

#### Purpose
* Handle late-arriving data
* Bound state size in aggregations
* Define how late data can arrive

#### Syntax
```python
df.withWatermark("timestamp_column", "threshold")
```

#### How It Works
```python
df.withWatermark("event_time", "10 minutes")
  .groupBy(window(col("event_time"), "5 minutes"))
  .count()
```
* Watermark = max(event_time) - 10 minutes
* Windows are kept open until watermark passes
* Events older than watermark are dropped
* Prevents unbounded state growth

#### Watermark Guarantees
* **Events within threshold**: Always processed
* **Events beyond threshold**: May be dropped
* **State cleanup**: Happens after watermark passes

---

### Stateful Operations

#### Aggregations
```python
.groupBy("key").agg(count("*"), sum("value"))
```
* Requires watermark for bounded state
* State grows with number of keys

#### Joins

**Stream-Static Join**
```python
stream_df.join(static_df, "key")
```
* No state required
* Static table loaded once

**Stream-Stream Join**
```python
stream1.withWatermark("time1", "10 min").join(
    stream2.withWatermark("time2", "10 min"),
    expr("key1 = key2 AND time1 >= time2 AND time1 <= time2 + interval 5 minutes")
)
```
* Requires watermarks on both sides
* Requires time-bound join condition
* State bounded by watermark + time constraint

#### Deduplication
```python
# Unbounded (not recommended)
df.dropDuplicates(["id"])

# Bounded (with watermark)
df.withWatermark("timestamp", "1 hour")
  .dropDuplicates(["id", "timestamp"])
```

---

### Checkpointing

#### Purpose
* Fault tolerance
* Exactly-once processing
* Track processing progress
* State recovery

#### Checkpoint Location
```python
.option("checkpointLocation", "/path/to/checkpoint")
```

#### What's Stored
* Offsets (where to resume)
* State (aggregations, joins)
* Metadata (query configuration)

#### Important Notes
* **Required** for production streams
* **Cannot change** checkpoint location
* **Delete** to restart from beginning
* **Unique** per streaming query

---

### Monitoring & Management

#### Query Status
```python
query.id              # Unique query ID
query.runId           # Current run ID
query.name            # Query name
query.isActive        # Is query running?
query.status          # Current status dict
```

#### Progress Metrics
```python
query.lastProgress    # Last batch metrics
query.recentProgress  # Recent batches
```

**Metrics Include**:
* Batch ID and timestamp
* Number of input rows
* Processing rate
* Input rate
* Source/sink descriptions
* State statistics

#### Query Control
```python
query.stop()                    # Stop the query
query.awaitTermination()        # Wait indefinitely
query.awaitTermination(timeout) # Wait with timeout
query.exception()               # Get exception if failed
```

#### List Active Queries
```python
spark.streams.active  # List all active queries
```

---

### Best Practices

#### Development
✅ **Start with rate source** — Easy testing  
✅ **Use console sink for debugging** — Inspect data flow  
✅ **Test with small data** — Validate logic first  
✅ **Add watermarks early** — Prevent unbounded state  
✅ **Use unique checkpoint locations** — Per query  

#### Performance
✅ **Choose right trigger mode** — Balance latency/throughput  
✅ **Filter early** — Reduce data volume  
✅ **Partition appropriately** — Parallelize processing  
✅ **Monitor state size** — Watch for growth  
✅ **Use Delta for sinks** — ACID guarantees  

#### Production
✅ **Always use checkpoints** — Fault tolerance  
✅ **Set appropriate watermarks** — Handle late data  
✅ **Monitor query metrics** — Processing rates  
✅ **Handle exceptions** — Graceful failures  
✅ **Use trigger once for batch** — Cost efficiency  
✅ **Test recovery** — Restart from checkpoint  

#### Anti-Patterns
❌ **No watermark with aggregations** — Unbounded state  
❌ **Ignoring late data** — Data loss  
❌ **Large batch intervals** — High latency  
❌ **No monitoring** — Silent failures  
❌ **Changing schema without restart** — Incompatibility  
❌ **Reusing checkpoint locations** — State corruption  

---

### Common Patterns

#### ETL Pipeline
```python
df = spark.readStream.format("cloudFiles").load("/input")
transformed = df.select(...).filter(...)
transformed.writeStream
    .format("delta")
    .option("checkpointLocation", "/checkpoint")
    .start("/output")
```

#### Real-time Aggregation
```python
df.withWatermark("timestamp", "10 minutes")
  .groupBy(window("timestamp", "5 minutes"), "key")
  .agg(count("*").alias("count"))
  .writeStream.outputMode("update").start()
```

#### Deduplication
```python
df.withWatermark("timestamp", "1 hour")
  .dropDuplicates(["id"])
  .writeStream.outputMode("append").start()
```

#### Late Data Handling
```python
df.withWatermark("event_time", "30 minutes")
  .groupBy(window("event_time", "10 minutes"))
  .count()
```

---

### Troubleshooting

#### Query Not Processing Data
* Check source has new data
* Verify checkpoint location is writable
* Check trigger interval
* Look at `query.status`

#### State Growing Unbounded
* Add watermark to aggregations
* Check watermark is advancing
* Verify time column is correct
* Consider shorter watermark threshold

#### High Latency
* Reduce trigger interval
* Increase parallelism
* Optimize transformations
* Check for data skew

#### Data Loss
* Verify exactly-once semantics
* Check checkpoint health
* Look for exceptions in logs
* Verify watermark settings

---

### Additional Resources

* **Databricks Documentation**: Structured Streaming Guide
* **Apache Spark**: Structured Streaming Programming Guide
* **Delta Lake**: Delta Streaming
* **Auto Loader**: Scalable file ingestion
* **Kafka Integration**: Kafka Structured Streaming